In [12]:
import pickle
import pandas as pd
import os

In [2]:
with open('model.bin', 'rb') as f:
    dv, model = pickle.load(f)

/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/codespace/anaconda3/envs/mlflow_conda/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
categorical = ['PULocationID', 'DOLocationID']
def read_data(filename):
    df = pd.read_parquet(filename)
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()
    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    return df

In [4]:
year = 2023
month = 3
df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet')

In [5]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

## Q1. what the standard deviation of the predicted duration for this dataset?

In [6]:
print(f"Standard deviation:{df['duration'].std():.2f}")

Standard deviation:10.60


## Q2. Preparing the output

In [ ]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')
df.to_parquet(f'data/yellow_tripdata_{year}-{month:02d}.parquet', engine='pyarrow', compression=None, index=False)
file_size = os.path.getsize(f'data/yellow_tripdata_{year}-{month:02d}.parquet') / 1024**2
print(f"File size: {file_size:.2f} MB")

File size: 147.46 MB


: 